# Minimum safety factor - verification

Use the **Python (FAITH labelmaker)** kernel. Set `shot`, drag a time range on
any panel, then press *Mark present* / *Mark absent*, *Verify* and *Save*.

`qmin` is the offline EFIT01 `aeqdsk:qmin` standing in for `qmin_EFITRT2`, and
every shot in this category's roster has it. The classes are the `qmin_rule`
ones: 0 absent, 1 low, 2 hybrid, 3 elevated, 4 high, with thresholds at
qmin > 0.95 (hybrid), > 1.5 (elevated) and > 2 (high) - the three dashed lines
on the first panel.

What to check: that the flat-top is where the rule says it is, and that a
class change sits on a real qmin crossing rather than on an EFIT excursion of
one or two frames.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np

from labeler.config import Paths
from labeler.features.store import read_feature
from labeler.events.verify import Panel, review

event = "minimum_safety_factor"
shot = 1  # replace with a shot from shots.csv
source = "extend_qmin_rule/recommender_v1"

features = Paths.from_env().features_file(shot)

qmin = read_feature(features, "qmin")
qpsi = read_feature(features, "qpsi")
ip = read_feature(features, "ip")

panels = [
    Panel(
        title="qmin (EFIT01 aeqdsk)",
        x=qmin.x * 1000.0,
        y=qmin.y,
        ylabel="q",
        # the rule's class thresholds, as three thin shaded lines
        bands=[(0.95, 0.96), (1.5, 1.51), (2.0, 2.01)],
    ),
    Panel(
        title="q profile",
        kind="heatmap",
        x=qpsi.x * 1000.0,
        y=np.linspace(0.0, 1.0, qpsi.y.shape[0]),
        z=qpsi.y,
        ylabel="rho",
    ),
    Panel(title="ip", x=ip.x * 1000.0, y=ip.y, ylabel="A"),
]

In [ ]:
session = review(event, shot, panels, source=source)
session

`category` on a correction is the q-min class, not a binary flag:
0 absent, 1 low, 2 hybrid, 3 elevated, 4 high. *Mark present* writes 1, so for
any other class call `session.mark(t_start, t_end, category=3)` directly.

```python
from labeler.events.verify import read_corrections, review_path

read_corrections(review_path(event, shot))
```
